<a href="https://colab.research.google.com/github/BaccarAla90/lunar-lander-ppo-from-scratch/blob/main/PPO_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gymnasium as gym
import torch
from model import ActorCritic
from memory import RolloutBuffer
from ppo import PPOAgent
from plot_tools import *
import numpy as np
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware engine connected: {device.type.upper()}\n")

In [ ]:
pip install swig

In [ ]:
pip install "gymnasium[box2d]"

# Training PPO: Clipped Objective Function

In [ ]:
env = gym.make("LunarLander-v3")  # Or "LunarLander-v3" depending on your library version

In [ ]:
history = {
    "ep_rewards": [],
    "actor_loss": [],
    "critic_loss": [],
    "entropy": []
}

# 1. Initialize the environment, memory buffer, and PPO agent
agent = PPOAgent(state_dim=8, action_dim=4, lr=0.0003)
memory = RolloutBuffer()
agent.policy.to(device)
max_training_episodes = 50
timestep_per_rollout = 8192
total_steps = 0

state, _ = env.reset()
episode_steps=0
for episode in range(max_training_episodes):
    episode_reward = 0

    # =========================================================================
    # PHASE 1: COLLECTING DATA (The Rollout Loop)
    # The neural network weights are FROZEN here. We are just recording a diary.
    # =========================================================================

    for step in range(timestep_per_rollout):
        total_steps += 1
        episode_steps+=1
        # Convert the raw 8-number game state into a PyTorch tensor
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

        # Ask model.py for an action, its log probability, and the critic's guess V(s)
        with torch.no_grad():
            action, log_prob, _, value = agent.policy.get_action_and_value(state_tensor, action=None)

        # Send the action integer to the LunarLander game engine
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        # SAVE EVERYTHING INTO THE MEMORY BUFFER (memory.py)
        memory.states.append(state_tensor)
        memory.actions.append(action)
        memory.logprobs.append(log_prob)  # <--- Saving pi_old here!
        memory.rewards.append(reward)
        memory.values.append(value.item())  # <--- Saving Critic's guess V(s) here!
        memory.is_terminals.append(float(done))  # <--- Saving crash/done flag here!

        state = next_state
        episode_reward += reward
        if terminated or truncated:
            history["ep_rewards"].append(episode_reward)  # Log BEFORE resetting
        # If the ship crashes or successfully lands, reset the environment for a new flight
        if done:
            state, _ = env.reset()
            print(f"Episode Done | Reward: {episode_reward:.2f} | Length: {episode_steps} steps")
            episode_reward = 0
            episode_steps=0

    # =========================================================================
    # PREPARING THE BOUNDARY VALUE (Scenario B)
    # The 2048-step timer hit. We must pause the game.
    # We ask the Critic for ONE final guess of the paused state to act as 'next_value' since we need it to compute GAE.
    # =========================================================================
    with torch.no_grad():
        last_state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        _, _, _, next_value = agent.policy.get_action_and_value(last_state_tensor)
        next_value = next_value.item()

    # =========================================================================
    # PHASE 2: OPTIMIZATION (Calling agent.update from ppo.py)
    # Hand the 2048-step diary and next_value over to ppo.py to run GAE and K Epochs!
    # =========================================================================
    print("Paused game. Running PPO optimization epochs...")
    a_loss, c_loss, ent = agent.update(memory, next_value, device)
    print("training_episode:",episode)
    history["actor_loss"].append(a_loss)
    history["critic_loss"].append(c_loss)
    history["entropy"].append(ent)

    print(f"--- Completed Training Iteration. Total Global Steps: {total_steps} ---\n")
torch.save(agent.policy.state_dict(), "ppo_lunar_lander_v3.pth")


In [ ]:
plot_training(history)

# Evaluation

In [ ]:
# (Make sure to use the exact filename you saved it as!)
test_agent = PPOAgent(state_dim=8, action_dim=4)
test_agent.policy.load_state_dict(torch.load("ppo_lunar_lander_v3.pth"))

# Put the network in evaluation mode (turns off exploration)
test_agent.policy.eval()
# Move it to the GPU if you are using one
test_agent.policy.to(device)

# 2. Setup the test environment
test_env = gym.make("LunarLander-v3")

print("Starting Test Flights from Saved File...")
for episode in range(5):
    state, _ = test_env.reset()
    done = False
    test_reward = 0

    while not done:
        # Convert state to tensor
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

        # 3. USE THE TEST_AGENT HERE!
        with torch.no_grad():
            action, _, _, _ = test_agent.policy.get_action_and_value(state_tensor)

        # Take the action in the environment
        state, reward, terminated, truncated, _ = test_env.step(action.item())
        done = terminated or truncated
        test_reward += reward

    print(f"Test Flight {episode + 1} - Score: {test_reward:.2f}")

test_env.close()

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import glob
import io
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay

# 1. Setup the recording environment
# We must set render_mode="rgb_array" so the wrapper can capture the frames!
test_env = gym.make("LunarLander-v3", render_mode="rgb_array")
test_env = RecordVideo(test_env, video_folder='./video', episode_trigger=lambda x: True)

# Put the agent in evaluation mode
test_agent.policy.eval()

# 2. Play exactly 1 game
state, _ = test_env.reset()
done = False
total_reward = 0

print("Agent is flying... recording video...")
while not done:
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        action, _, _, _ = test_agent.policy.get_action_and_value(state_tensor)

    state, reward, terminated, truncated, _ = test_env.step(action.item())
    done = terminated or truncated
    total_reward += reward

test_env.close()
print(f"Test Flight Complete! Final Score: {total_reward:.2f}")

# 3. Display the video inside Colab
video_files = glob.glob('./video/*.mp4')
if len(video_files) > 0:
    # Get the latest video file
    latest_video = max(video_files, key=lambda x: x)
    mp4 = open(latest_video, 'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()

    ipythondisplay.display(HTML(f"""
    <video width="640" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("Could not find the video file.")